In [82]:
import pandas as pd 

MainAtApogee_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/Polaris_Cycle1_Aug2025-MainAtApogee.csv')
MainAtApogee_df = MainAtApogee_df.set_index('Simulation')

DrogueOnly_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/Polaris_Cycle1_Aug2025-DrogueOnly.csv')
DrogueOnly_df = DrogueOnly_df.set_index('Simulation')

Ballistic_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/Polaris_Cycle1_Aug2025-Ballistic.csv')
Ballistic_df = Ballistic_df.set_index('Simulation')



In [83]:
SimParams_df = pd.read_csv('/Users/ricechessmaster/Documents/Dispersion Analysis v2/backend/data/sim_parameters_historical_aug2025_filtered.csv')
print(SimParams_df['date'].head())

0    2025-08-01 11:00:00
1    2025-08-01 12:00:00
2    2025-08-01 13:00:00
3    2025-08-01 14:00:00
4    2025-08-01 15:00:00
Name: date, dtype: str


In [85]:
# Order of when to join would matter eventually if there is too much data; could apply where first, then join
# left_index = True is used as the name "Simulations" for the 0th column is not real... it's a default

Full_MainAtApogee_df = pd.merge(MainAtApogee_df, SimParams_df, left_index=True, right_on='date', how="inner")
Full_DrogueOnly_df = pd.merge(DrogueOnly_df, SimParams_df, left_index=True, right_on='date', how="inner")
Full_Ballistic_df = pd.merge(Ballistic_df, SimParams_df, left_index=True, right_on='date', how="inner")


In [92]:
import numpy as np


def haversine_nm(ref_latitude:float, ref_longitude:float, latitude, longitude):
    """
    Compute the great-circle distance between two points on Earth (in nautical miles).
    """

    # Earth's mean radius in nautical miles
    R_nm = 3443.92

    # Note: input of 2 scalars and 2 arrays to be converted to radians
    ref_latitude, ref_longitude, latitude, longitude = map(np.radians, [ref_latitude, ref_longitude, latitude, longitude])

    dlat = latitude - ref_latitude
    dlon = longitude - ref_longitude

    a = np.sin(dlat / 2) ** 2 + np.cos(ref_latitude) * np.cos(latitude) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R_nm * c


# reference launch coordinates (degrees)
ref_lat = 47.965378
ref_long = -81.873536

Main_landing_latitudes = Full_MainAtApogee_df["Polaris Rocket Landing Latitude (°N)"].to_numpy()
Main_landing_longitudes = Full_MainAtApogee_df["Polaris Rocket Landing Longitude (°E)"].to_numpy()

# compute great-circle distance in nautical miles using the shared haversine helper
landing_distances = haversine_nm(ref_lat, ref_long, Main_landing_latitudes, Main_landing_longitudes)

Full_MainAtApogee_df['distance_nm'] = landing_distances
Main_safety_column = np.where(landing_distances < 10, True, False)
Full_MainAtApogee_df["safety_column"] = Main_safety_column
# print(Full_MainAtApogee_df['distance_nm'].describe())

In [93]:
DrogueOnly_landing_latitudes = Full_DrogueOnly_df["Polaris Rocket Landing Latitude (°N)"].to_numpy()
DrogueOnly_landing_longitudes = Full_DrogueOnly_df["Polaris Rocket Landing Longitude (°E)"].to_numpy()

# compute great-circle distance in nautical miles using the shared haversine helper
landing_distances = haversine_nm(ref_lat, ref_long, DrogueOnly_landing_latitudes, DrogueOnly_landing_longitudes)

# insert distances and safety column into dataframe
Full_DrogueOnly_df['distance_nm'] = landing_distances
DrogueOnly_safety_column = np.where(landing_distances < 10, True, False)
Full_DrogueOnly_df["safety_column"] = DrogueOnly_safety_column

In [94]:
# Extract and use the haversine function to compute distance from the starting point
Ballistic_landing_latitudes = Full_Ballistic_df["Polaris Rocket Landing Latitude (°N)"].to_numpy()
Ballistic_landing_longitudes = Full_Ballistic_df["Polaris Rocket Landing Longitude (°E)"].to_numpy()

# compute great-circle distance in nautical miles using the shared haversine helper
landing_distances = haversine_nm(ref_lat, ref_long, Ballistic_landing_latitudes, Ballistic_landing_longitudes)

# insert distances and safety column into dataframe
Full_Ballistic_df['distance_nm'] = landing_distances
Ballistic_safety_column = np.where(landing_distances < 10, True, False)
Full_Ballistic_df["safety_column"] = Ballistic_safety_column

In [95]:
Full_safety_col_arry = Full_MainAtApogee_df["safety_column"].to_numpy()
DrogueOnly_safety_col_arry = Full_DrogueOnly_df["safety_column"].to_numpy()
Ballistic_safety_col_arry = Full_Ballistic_df["safety_column"].to_numpy()

# Default axis to None, which would flatten the array either way
safe, unsafe  = np.count_nonzero(Full_safety_col_arry == True), np.count_nonzero(Full_safety_col_arry == False)
print(safe, unsafe)

safe, unsafe  = np.count_nonzero(DrogueOnly_safety_col_arry == True), np.count_nonzero(DrogueOnly_safety_col_arry == False)
print(safe, unsafe)

safe, unsafe  = np.count_nonzero(Ballistic_safety_col_arry == True), np.count_nonzero(Ballistic_safety_col_arry == False)
print(safe, unsafe)


227 1
228 0
228 0


In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

column_names = ["temperature", "pressure", "Max Windspeed (mph)"]
altitudes = [110, 320, 500, 800, 1000, 1500, 1900, 3200, 4200, 5600, 7200, 9200, 10400, 11800, 13500, 15800, 17700, 19300, 22000]

for alt in altitudes:
    column_names.append(str(alt)) 
    column_names.append(f"direction [{str(alt)}]") 

# Define the input and the output 
X = Full_MainAtApogee_df[column_names]
y = Full_MainAtApogee_df['distance_nm']



#landing_lat: Mapped[float] = mapped_column(nullable=False)
#landing_lon: Mapped[float] = mapped_column(nullable=False)
#distance_nm: Mapped[float] = mapped_column(nullable=False)



# main_model = XGBRegressor()


# train_test_split()

IndentationError: unexpected indent (788603984.py, line 9)